# TITANIC

- ??
- ?? ????
- ?? ??
    - ??? ????
- ???? ?? ??
- ?? => ??
- ??? (CSV)

In [ ]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

In [ ]:
train = pd.read_csv("./data/titanic_data/titanic_train.csv")
test = pd.read_csv("./data/titanic_data/titanic_test.csv")

### ??? ??

In [ ]:
print(train.info())
train.head(4)

### ???

In [ ]:
print(sorted(train["Fare"].unique()))

In [ ]:
print(train.isnull().sum())
print(test.isnull().sum())

In [ ]:
print(train["Embarked"].unique())
print(train["Embarked"].value_counts()) 
# ???? S ??? Embarked ??? S ??
train["Embarked"] = train["Embarked"].fillna("S")

In [ ]:
# ??? ???? ?? ???? ??? ??? ???? ???
train["Age"] = train["Age"].fillna(train["Age"].median())
test["Age"] = test["Age"].fillna(test["Age"].median())
test["Fare"] = test["Fare"].fillna(test["Fare"].median())

In [ ]:
print(train.isnull().sum())
print(test.isnull().sum())

### ??? ??

In [ ]:
train["Sex"][train["Sex"]=="male"] = 0
train["Sex"][train["Sex"]=="female"] = 1
test["Sex"][test["Sex"]=="male"] = 0
test["Sex"][test["Sex"]=="female"] = 1
train["Embarked"][train["Embarked"]=="S"] = 0
train["Embarked"][train["Embarked"]=="C"] = 1
train["Embarked"][train["Embarked"]=="Q"] = 2
test["Embarked"][test["Embarked"]=="S"] = 0
test["Embarked"][test["Embarked"]=="C"] = 1
test["Embarked"][test["Embarked"]=="Q"] = 2

## ??? ???

In [ ]:
X = train.drop(["Survived","Name","Ticket","Cabin"],axis=1)
y = train["Survived"]

In [ ]:
X,y

[xgboost](https://github.com/dmlc/xgboost)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, random_state=42)

In [ ]:
X_train = X_train.astype("float")
X_test = X_test.astype("float")
y_train = y_train.astype("float")
y_test = y_test.astype("float")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb

In [ ]:
# ?? ??
# X_train.info()
# X_test.info()
# y_train.info()
# y_test.info()

In [ ]:
# xgboost colsample, subsample ?? ??
params = {"objective":["binary:logistic"],                              # ??, ???? (???? ??? ?????? ??? ??)
          "booster":["gbtree"],                                         # gbtree ????(??? ??)
          "eta":[0.01],                                                 # ??????, ???? ????
          "max_depth":[0,1,2,3,4,5,6,7,8,9,10],                         # 1024 ?? ??? 2^10
          "min_child_weight":[0,1,2,3,4,5,6,7,8,9,10],                  # ???
          "colsample_bytree":[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0], # 0.1??? ??? 1.0 ??
          "subsample":[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0],        # 0.1??? ???
          "alpha":[0,1,2,3,4,5],
          "lambda":[0,1,2,3,4,5],
          "n_estimators":[50000], # ?? ???(?? dic?? ??? ??? ?) 5? ??(??? ???? ??)
          "random_state":[42]
         }

In [ ]:
clf = xgb.XGBClassifier() #default
clf_grid = RandomizedSearchCV(clf,
                              params,
                              cv=5,      # cross_val 5? 
                              n_iter=50, # ?? ??? 50
                              random_state=42,
                              scoring="accuracy")
# 5*50*50000?

In [ ]:
# ??? ??? ?? 50??? ?? ?? ??? ??? ???? (TradeOff)
clf_grid.fit(X_train, y_train,
             early_stopping_rounds=50,
             eval_set = [(X_test, y_test)], # eval ??? test case
             eval_metric = "error",
             verbose=0)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
print(clf_grid.best_params_)
print(clf_grid.best_score_)

In [ ]:
bst = clf_grid.best_estimator_
print(bst)

In [ ]:
pred_1 = bst.predict(X_test)
grid_score = accuracy_score(y_test, pred_1)
print(grid_score)

In [ ]:
confusion_matrix(y_test, pred_1)

In [ ]:
import matplotlib.pyplot as plt
def plot_f_importances(model):
    n_f = X.shape[1]
    plt.barh(range(n_f), model.feature_importances_, align="center")
    plt.yticks(np.arange(n_f),X)
    plt.show()

In [ ]:
plot_f_importances(bst)
# Sex? ?? ?? => EDA? ??? ?????

In [ ]:
X.keys()

In [ ]:
X = test.loc[:,['PassengerId', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked']]
X = X.astype("float")
Y = bst.predict(X)
Y = np. array(Y, dtype = "int64")
print(Y)

In [ ]:
PassangerId = np.array(test["PassengerId"]).astype(int)
result = pd.DataFrame(Y, PassangerId, columns=["Survived"])

In [ ]:
result.to_csv("submission.csv", index_label=["PassengerId"])

##### Ticket

In [ ]:
train["Ticket"].dtype